# 일본어 애니메이션: 영상 → 대사·시간·화자

Qwen3-ASR 1.7B → Qwen3-ForcedAligner 0.6B → pyannote Community-1을 순서대로 실행합니다.

Colab에서 `런타임 → 런타임 유형 변경 → T4 GPU`를 선택하고 셀을 하나씩 실행하세요. 설치 후에는 세션을 한 번 다시 시작합니다. 모든 모델을 동시에 올리지 않고 단계마다 해제합니다.

[Community-1](https://huggingface.co/pyannote/speaker-diarization-community-1) 이용 조건에 동의하고 [Hugging Face 읽기 토큰](https://huggingface.co/settings/tokens)을 준비하세요. Colab 왼쪽 열쇠 메뉴에 `HF_TOKEN`을 저장하고 접근을 허용하거나, 인증 셀의 숨김 입력창에서 입력합니다.

결과는 Drive에 저장합니다. 화자는 `SPEAKER_00` 등의 번호이며, 마지막 셀에서 대표 음성을 들을 수 있습니다. 캐릭터 이름의 자동 식별은 포함하지 않습니다. 애니메이션의 작은 목소리·비명·노래·겹친 대사는 오류가 날 수 있습니다.

재개할 때는 셀 4가 출력한 폴더명을 `RUN_NAME`에 입력하세요. 새 런타임에서는 설치·세션 재시작을 마친 뒤 셀 3부터 실행합니다. 완료된 전사·정렬 구간을 재사용합니다. 화자 분리는 완료 결과가 있을 때 재사용하며, 처리 도중 중단되면 해당 단계 전체를 다시 실행합니다.

## 셀 1 — 실행 환경·패키지 설치

Colab 런타임에 설치합니다. GPU 패키지는 호환 조합으로 고정했고 Qwen-ASR 0.0.6의 Transformers 4.57.6 요구 조건을 적용했습니다. 다운로드가 끝날 때까지 기다리세요.

In [ ]:
import sys, subprocess, shutil
from pathlib import Path
if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError(f"지원 Python은 3.10~3.13입니다. 현재: {sys.version.split()[0]}")
subprocess.run(["nvidia-smi"], check=True)
if not shutil.which("ffmpeg"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
uv = shutil.which("uv")
if not uv:
    subprocess.run(["bash", "-c", "curl -LsSf https://astral.sh/uv/install.sh | sh"], check=True)
    uv = str(Path.home() / ".local/bin/uv")
constraints = Path("/content/stt_constraints.txt")
constraints.write_text("torch==2.8.0\ntorchaudio==2.8.0\ntorchvision==0.23.0\ntorchcodec==0.7.0\ntransformers==4.57.6\n")
base = [uv, "pip", "install", "--system", "--python", sys.executable]
subprocess.run(base + ["--index-url", "https://download.pytorch.org/whl/cu126",
    "torch==2.8.0", "torchaudio==2.8.0", "torchvision==0.23.0"], check=True)
subprocess.run(base + ["-c", str(constraints), "qwen-asr==0.0.6", "pyannote.audio==4.0.7",
    "torchcodec==0.7.0", "silero-vad==6.2.3", "pandas", "soundfile", "tqdm"], check=True)
print("설치 완료. 세션을 다시 시작한 뒤 셀 3부터 진행하세요.")

## 셀 2 — 세션 다시 시작

메뉴에서 `런타임 → 세션 다시 시작`을 선택하세요. 이 안내에는 실행 코드가 없습니다. 이후 설치 셀을 반복하지 않고 셀 3부터 실행합니다.

## 셀 3 — 라이브러리·GPU 확인·Drive 연결

import 오류는 설치 또는 세션 재시작 단계에서 확인하세요. GPU가 없으면 중단합니다.

In [ ]:
import gc, json, math, subprocess, shutil
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch, torchaudio, soundfile as sf
from tqdm.auto import tqdm
from IPython.display import display, Audio
from qwen_asr import Qwen3ASRModel, Qwen3ForcedAligner
from pyannote.audio import Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook
from silero_vad import load_silero_vad, get_speech_timestamps
from google.colab import drive
if not torch.cuda.is_available():
    raise RuntimeError("GPU가 없습니다. 런타임 유형을 T4 GPU로 바꾸세요.")
print("GPU:", torch.cuda.get_device_name(0))
for package in ["torch", "torchaudio", "qwen-asr", "transformers", "pyannote.audio", "silero-vad"]:
    print(package, version(package))
DEVICE, DTYPE = "cuda:0", torch.float16
drive.mount("/content/drive")

## 셀 4 — 영상 경로·처리 설정

`VIDEO_PATH`를 실제 Drive 파일 경로로 바꾸세요. 첫 실행은 `RUN_NAME=""`, 재개는 기존 결과 폴더명을 입력합니다. 설정을 바꿔 재처리하면 새 결과 폴더를 사용하세요. 화자 수는 자동 추정하며 14명으로 고정하지 않습니다.

In [ ]:
VIDEO_PATH = "/content/drive/MyDrive/여기에_영상_경로.mp4"
OUTPUT_ROOT = "/content/drive/MyDrive/STT_results"
RUN_NAME = ""  # 재개할 때 기존 결과 폴더명
AUDIO_TRACK = 0  # 다음 셀에서 확인하는 오디오 트랙 순번
ASR_MODEL = "Qwen/Qwen3-ASR-1.7B"
ALIGN_MODEL = "Qwen/Qwen3-ForcedAligner-0.6B"
DIAR_MODEL = "pyannote/speaker-diarization-community-1"
MAX_SPEECH_SECONDS = 25.0
VAD_THRESHOLD = 0.4
MIN_SPEECH_MS, MIN_SILENCE_MS, SPEECH_PAD_MS = 100, 400, 150
MAX_NEW_TOKENS = 768
NUM_SPEAKERS = None  # 정확한 실제 화자 수를 알 때만 지정
MIN_SPEAKERS, MAX_SPEAKERS = None, None
DIALOGUE_GAP_SECONDS, DIALOGUE_MAX_SECONDS = 0.8, 8.0
ALIGN_TOLERANCE_SECONDS = 0.15
MIN_SPEAKER_COVERAGE = 0.5
SR = 16000
video_path = Path(VIDEO_PATH)
if not video_path.is_file():
    raise FileNotFoundError(f"영상 경로를 확인하세요: {video_path}")
if NUM_SPEAKERS is not None and (MIN_SPEAKERS is not None or MAX_SPEAKERS is not None):
    raise ValueError("NUM_SPEAKERS와 MIN/MAX_SPEAKERS를 동시에 지정하지 마세요.")
if not 0 < MAX_SPEECH_SECONDS <= 30 or not 0 < VAD_THRESHOLD < 1:
    raise ValueError("MAX_SPEECH_SECONDS는 0~30초, VAD_THRESHOLD는 0~1 사이로 설정하세요.")
if not RUN_NAME:
    RUN_NAME = video_path.stem + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
if Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
    raise ValueError("RUN_NAME에는 폴더명 하나만 입력하세요.")
run_dir = Path(OUTPUT_ROOT) / RUN_NAME
work_dir = Path("/content/stt_work") / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
work_dir.mkdir(parents=True, exist_ok=True)
wav_path = work_dir / "audio_16k.wav"
stat = video_path.stat()
settings = {"video": str(video_path), "video_size": stat.st_size, "video_mtime": stat.st_mtime,
    "audio_track": AUDIO_TRACK, "asr": ASR_MODEL, "align": ALIGN_MODEL, "diar": DIAR_MODEL,
    "max_speech_seconds": MAX_SPEECH_SECONDS, "vad_threshold": VAD_THRESHOLD,
    "min_speech_ms": MIN_SPEECH_MS, "min_silence_ms": MIN_SILENCE_MS, "speech_pad_ms": SPEECH_PAD_MS,
    "max_new_tokens": MAX_NEW_TOKENS, "num_speakers": NUM_SPEAKERS,
    "min_speakers": MIN_SPEAKERS, "max_speakers": MAX_SPEAKERS,
    "dialogue_gap": DIALOGUE_GAP_SECONDS, "dialogue_max": DIALOGUE_MAX_SECONDS,
    "align_tolerance": ALIGN_TOLERANCE_SECONDS, "min_speaker_coverage": MIN_SPEAKER_COVERAGE,
    "packages": {p: version(p) for p in ["qwen-asr", "pyannote.audio", "silero-vad"]}}
def save_json(path, value):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(path)
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))
settings_path = run_dir / "settings.json"
if settings_path.exists() and read_json(settings_path) != settings:
    raise RuntimeError("기존 결과와 영상 또는 설정이 다릅니다. 새 RUN_NAME으로 시작하세요.")
save_json(settings_path, settings)
print("결과 폴더:", run_dir)
print("재개용 RUN_NAME:", RUN_NAME)

## 셀 5 — 오디오 트랙 확인

`audio_track`은 셀 4의 `AUDIO_TRACK`에 넣는 번호입니다. 일본어 트랙인지 확인하세요. 번호를 바꾸려면 셀 4에서 `RUN_NAME=""`로 새 폴더를 만들고 이 셀을 다시 실행합니다.

In [ ]:
probe = subprocess.run(["ffprobe", "-v", "error", "-show_streams", "-show_format", "-of", "json", str(video_path)],
    check=True, capture_output=True, text=True)
media = json.loads(probe.stdout)
audio_streams = [s for s in media["streams"] if s["codec_type"] == "audio"]
if not audio_streams:
    raise RuntimeError("영상에 오디오 트랙이 없습니다.")
rows = [{"audio_track": i, "stream_index": s["index"], "codec": s.get("codec_name"),
    "channels": s.get("channels"), "language": s.get("tags", {}).get("language", "미표기"),
    "title": s.get("tags", {}).get("title", "")} for i, s in enumerate(audio_streams)]
display(pd.DataFrame(rows))
if not isinstance(AUDIO_TRACK, int) or not 0 <= AUDIO_TRACK < len(audio_streams):
    raise ValueError("AUDIO_TRACK 번호가 범위를 벗어났습니다.")
print("선택 트랙:", rows[AUDIO_TRACK])
save_json(run_dir / "media_info.json", media)

## 셀 6 — 영상 복사·음성 추출·미리 듣기

영상 시작에 맞춰 16kHz 모노 WAV를 추출합니다. BGM과 효과음도 남습니다. 언어와 트랙을 직접 들어 확인하세요. `PREVIEW_START_SECONDS`를 바꾸면 다른 구간을 재생합니다.

In [ ]:
PREVIEW_START_SECONDS, PREVIEW_SECONDS = 60.0, 15.0
local_video = work_dir / ("source" + video_path.suffix)
if not wav_path.exists():
    if not local_video.exists() or local_video.stat().st_size != video_path.stat().st_size:
        shutil.copy2(video_path, local_video)
    temp_wav = work_dir / "audio_16k.partial.wav"
    process = subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-copyts", "-start_at_zero",
        "-i", str(local_video), "-map", f"0:a:{AUDIO_TRACK}", "-vn", "-af", "aresample=async=1:first_pts=0",
        "-ac", "1", "-ar", str(SR), "-c:a", "pcm_s16le", str(temp_wav)], capture_output=True, text=True)
    if process.returncode:
        raise RuntimeError("음성 추출 실패:\n" + process.stderr)
    temp_wav.replace(wav_path)
audio, sample_rate = sf.read(wav_path, dtype="float32")
if sample_rate != SR or audio.ndim != 1 or not len(audio) or not np.isfinite(audio).all():
    raise RuntimeError("추출 오디오의 형식 또는 데이터가 올바르지 않습니다.")
duration = len(audio) / SR
print(f"추출 완료: {duration/60:.2f}분 / {SR}Hz / 모노 / {wav_path}")
start = min(max(0.0, PREVIEW_START_SECONDS), max(0.0, duration-PREVIEW_SECONDS))
display(Audio(audio[int(start*SR):int((start+PREVIEW_SECONDS)*SR)], rate=SR))

## 셀 7 — 음성 구간 찾기

Silero VAD는 CPU에서 실행합니다. 작은 목소리가 빠지면 새 결과 폴더에서 `VAD_THRESHOLD`를 낮춰 다시 처리할 수 있습니다. 긴 구간은 최대 약 25초로 나눕니다. 첫 구간을 재생합니다.

In [ ]:
segments_path = run_dir / "speech_segments.json"
if segments_path.exists():
    speech_segments = read_json(segments_path)
else:
    vad_model = load_silero_vad()
    raw_segments = get_speech_timestamps(torch.from_numpy(audio), vad_model, sampling_rate=SR,
        threshold=VAD_THRESHOLD, min_speech_duration_ms=MIN_SPEECH_MS,
        max_speech_duration_s=MAX_SPEECH_SECONDS, min_silence_duration_ms=MIN_SILENCE_MS,
        speech_pad_ms=SPEECH_PAD_MS)
    speech_segments = [{"id": i, "start_sample": int(s["start"]), "end_sample": int(s["end"]),
        "start": s["start"]/SR, "end": s["end"]/SR} for i, s in enumerate(raw_segments)]
    del vad_model
    gc.collect()
    save_json(segments_path, speech_segments)
if not speech_segments:
    raise RuntimeError("음성 구간이 없습니다. 선택 오디오와 VAD 설정을 확인하세요.")
print(f"음성 구간 {len(speech_segments)}개 / 합계 {sum(s['end']-s['start'] for s in speech_segments)/60:.2f}분")
display(pd.DataFrame(speech_segments).head(10))
s = speech_segments[0]
display(Audio(audio[s["start_sample"]:s["end_sample"]], rate=SR))

## 셀 8 — STT 모델 불러오기

배치는 1, T4용 float16으로 실행합니다. 전사 결과가 모두 저장되어 있으면 모델 다운로드를 생략합니다.

In [ ]:
asr_dir = run_dir / "asr_chunks"
asr_dir.mkdir(exist_ok=True)
def chunk_path(directory, segment):
    return directory / f"{segment['id']:05d}.json"
if any(not chunk_path(asr_dir, s).exists() for s in speech_segments):
    asr_model = Qwen3ASRModel.from_pretrained(ASR_MODEL, dtype=DTYPE, device_map=DEVICE,
        attn_implementation="sdpa", max_inference_batch_size=1, max_new_tokens=MAX_NEW_TOKENS)
    print("STT 모델 준비 완료")
else:
    print("저장된 전사 결과가 있어 모델 로드를 생략했습니다.")
print(f"GPU 사용 메모리: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")

## 셀 9 — 첫 구간 대사 확인

일본어 대사와 셀 7의 음성을 비교한 뒤 전체 STT로 진행하세요. 여기서 실패하면 전체 처리 전에 멈춥니다.

In [ ]:
def transcribe_segment(segment):
    path = chunk_path(asr_dir, segment)
    if path.exists():
        return read_json(path)
    print(f"STT 구간 {segment['id']}: {segment['start']:.2f}~{segment['end']:.2f}초", flush=True)
    try:
        result = asr_model.transcribe(audio=(audio[segment["start_sample"]:segment["end_sample"]], SR),
            language="Japanese", return_time_stamps=False)[0]
        record = {**segment, "language": result.language, "text": result.text,
            "empty_transcript": not bool(result.text.strip())}
        save_json(path, record)
        return record
    except Exception as exc:
        save_json(run_dir / "last_error.json", {"stage": "STT", "segment": segment, "error_type": type(exc).__name__})
        raise RuntimeError(f"STT 실패: 구간 {segment['id']}, {segment['start']:.2f}~{segment['end']:.2f}초") from exc
first_transcript = transcribe_segment(speech_segments[0])
print("대사:", first_transcript["text"])
if first_transcript["empty_transcript"]:
    print("확인 필요: 첫 구간 전사가 비었습니다. 음성을 확인하세요.")

## 셀 10 — 전체 STT·구간별 저장

진행률과 구간 번호가 표시됩니다. 성공한 구간은 바로 Drive에 저장합니다. 실패하면 이 셀을 다시 실행할 때 완료 구간을 건너뜁니다. 빈 전사는 따로 표시됩니다.

In [ ]:
transcripts = []
for segment in tqdm(speech_segments, desc="일본어 STT"):
    record = transcribe_segment(segment)
    transcripts.append(record)
    if record["empty_transcript"]:
        print(f"확인 필요: 구간 {record['id']} 전사가 비었습니다.")
save_json(run_dir / "transcripts.json", transcripts)
print("STT 완료:", len(transcripts), "구간 / 빈 전사:", sum(r["empty_transcript"] for r in transcripts))
display(pd.DataFrame(transcripts)[["id", "start", "end", "text"]].head(15))

## 셀 11 — STT 모델 해제·정렬 모델 불러오기

STT 모델을 해제한 뒤 Qwen3-ForcedAligner 0.6B를 불러옵니다. 저장된 텍스트를 실제 음성과 맞추는 별도 단계입니다.

In [ ]:
if "asr_model" in globals():
    del asr_model
gc.collect()
torch.cuda.empty_cache()
align_dir = run_dir / "aligned_chunks"
align_dir.mkdir(exist_ok=True)
transcripts = read_json(run_dir / "transcripts.json")
if not any(r["text"].strip() for r in transcripts):
    raise RuntimeError("전사된 대사가 없습니다. 오디오와 STT 결과를 확인하세요.")
if any(r["text"].strip() and not chunk_path(align_dir, r).exists() for r in transcripts):
    align_model = Qwen3ForcedAligner.from_pretrained(ALIGN_MODEL, dtype=DTYPE, device_map=DEVICE,
        attn_implementation="sdpa")
    print("정렬 모델 준비 완료")
else:
    print("저장된 정렬 결과가 있어 모델 로드를 생략했습니다.")
print(f"GPU 사용 메모리: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")

## 셀 12 — 첫 대사 시간 정렬 확인

영상 시작 기준 초 단위입니다. 정렬 결과가 비었거나 모든 단어 길이가 0이면 대사를 보존하고 음성 구간 전체 시간을 임시 사용합니다. `alignment_failed=True`, `timestamp_source="speech_segment"`, `needs_review=True`로 표시합니다. 시간 역전·범위 이탈은 오류로 중단합니다. 일부 단어 길이가 0이면 `needs_review`에 표시합니다. 정렬에서 제거한 문장부호는 가능한 범위에서 원문으로 복원합니다.

In [ ]:
def align_segment(record):
    path = chunk_path(align_dir, record)
    if path.exists():
        return read_json(path)
    if not record["text"].strip():
        value = {"id": record["id"], "words": [], "empty_transcript": True}
        save_json(path, value)
        return value
    print(f"정렬 구간 {record['id']}: {record['start']:.2f}~{record['end']:.2f}초", flush=True)
    try:
        aligned = align_model.align(audio=(audio[record["start_sample"]:record["end_sample"]], SR),
            text=record["text"], language="Japanese")[0]
        words, previous_start = [], -1.0
        for item in aligned:
            start, end = float(item.start_time), float(item.end_time)
            if (not math.isfinite(start+end) or start < 0 or end < start
                or end > record["end"]-record["start"]+ALIGN_TOLERANCE_SECONDS or start < previous_start):
                raise ValueError(f"비정상 정렬: {item.text!r}, {start}~{end}")
            previous_start = start
            words.append({"segment_id": record["id"], "start": record["start"]+start,
                "end": record["start"]+end, "text": item.text, "display_text": item.text, "needs_review": end == start,
                "alignment_failed": False, "timestamp_source": "forced_alignment"})
        if not words or all(w["end"] == w["start"] for w in words):
            # 정확한 발화 시간을 얻지 못했으므로 음성 구간 전체를 임시 사용합니다.
            words = [{"segment_id": record["id"], "start": record["start"], "end": record["end"],
                "text": record["text"], "display_text": record["text"], "needs_review": True,
                "alignment_failed": True, "timestamp_source": "speech_segment"}]
            value = {"id": record["id"], "words": words, "empty_transcript": False,
                "alignment_failed": True, "failure_reason": "empty_or_zero_duration_alignment",
                "raw_alignment": [{"text": item.text, "start": float(item.start_time),
                    "end": float(item.end_time)} for item in aligned]}
            save_json(path, value)
            print(f"확인 필요: 구간 {record['id']} 정렬 실패 — "
                f"음성 구간 {record['start']:.2f}~{record['end']:.2f}초를 임시 사용합니다.")
            return value
        positions, cursor = [], 0
        for word in words:
            position = record["text"].find(word["text"], cursor)
            if position < 0:
                positions = []
                break
            positions.append(position)
            cursor = position+len(word["text"])
        if positions:
            for i, word in enumerate(words):
                left = 0 if i == 0 else positions[i]
                right = positions[i+1] if i+1 < len(words) else len(record["text"])
                word["display_text"] = record["text"][left:right]
        else:
            for word in words:
                word["needs_review"] = True
            print("확인 필요: 문장부호 복원 실패. 원문 전사 JSON을 확인하세요.")
        value = {"id": record["id"], "words": words, "empty_transcript": False}
        save_json(path, value)
        if any(w["needs_review"] for w in words):
            print(f"확인 필요: 구간 {record['id']}의 일부 정렬 결과")
        return value
    except Exception as exc:
        save_json(run_dir / "last_error.json", {"stage": "alignment", "segment_id": record["id"],
            "error_type": type(exc).__name__})
        raise RuntimeError(f"시간 정렬 실패: 구간 {record['id']}. 전사와 음성을 확인하세요.") from exc
first_nonempty = next(r for r in transcripts if r["text"].strip())
display(pd.DataFrame(align_segment(first_nonempty)["words"]))

## 셀 13 — 전체 시간 정렬·저장·모델 해제

실패 구간부터 재개할 수 있습니다. 단어 단위 결과를 저장한 뒤 정렬 모델을 해제합니다.

In [ ]:
words = []
for record in tqdm(transcripts, desc="대사 시간 정렬"):
    words.extend(align_segment(record)["words"])
save_json(run_dir / "aligned_words.json", words)
print(f"정렬 완료: {len(words)}개 / 확인 필요: {sum(w['needs_review'] for w in words)}개")
if "align_model" in globals():
    del align_model
gc.collect()
torch.cuda.empty_cache()
print(f"해제 후 GPU 사용 메모리: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")

## 셀 14 — Hugging Face 인증·화자 모델 불러오기

401/403 오류는 계정의 이용 조건 동의와 토큰의 모델 읽기 권한을 확인하세요. 토큰은 결과 파일에 저장하지 않습니다.

In [ ]:
diar_path = run_dir / "diarization.json"
if not diar_path.exists():
    from google.colab import userdata
    from getpass import getpass
    try:
        hf_token = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        hf_token = None
    if not hf_token:
        hf_token = getpass("Hugging Face 읽기 토큰: ")
    if not hf_token.strip():
        raise ValueError("HF_TOKEN이 비었습니다.")
    try:
        diar_pipeline = Pipeline.from_pretrained(DIAR_MODEL, token=hf_token.strip())
    finally:
        del hf_token
    if diar_pipeline is None:
        raise RuntimeError("Community-1 이용 조건과 토큰 권한을 확인하세요.")
    diar_pipeline.to(torch.device(DEVICE))
    print("화자 모델 준비 완료")
else:
    print("저장된 화자 분리 결과가 있어 모델 로드를 생략했습니다.")

## 셀 15 — 영상 전체 화자 분리

영상 안에서 번호를 연결하기 위해 전체 오디오를 한 번에 분석합니다. 진행률이 표시됩니다. 겹친 발화를 보존한 일반 결과와 대사 연결용 단일 화자 결과를 함께 저장합니다. 추정 화자 수가 실제 등장인물 수와 다를 수 있습니다.

In [ ]:
if diar_path.exists():
    diar_data = read_json(diar_path)
else:
    diar_options = {name: value for name, value in [("num_speakers", NUM_SPEAKERS),
        ("min_speakers", MIN_SPEAKERS), ("max_speakers", MAX_SPEAKERS)] if value is not None}
    try:
        with ProgressHook() as hook:
            diar_output = diar_pipeline({"waveform": torch.from_numpy(audio).unsqueeze(0), "sample_rate": SR},
                hook=hook, **diar_options)
        regular = [{"start": float(t.start), "end": float(t.end), "speaker": s}
            for t, _, s in diar_output.speaker_diarization.itertracks(yield_label=True)]
        exclusive = [{"start": float(t.start), "end": float(t.end), "speaker": s}
            for t, _, s in diar_output.exclusive_speaker_diarization.itertracks(yield_label=True)]
        if not exclusive:
            raise RuntimeError("화자 구간이 없습니다. 추출 음성을 확인하세요.")
        diar_data = {"regular": regular, "exclusive": exclusive}
        save_json(diar_path, diar_data)
        del diar_output
    except Exception as exc:
        save_json(run_dir / "last_error.json", {"stage": "diarization", "error_type": type(exc).__name__})
        raise RuntimeError("화자 분리 실패. 아래 원인 오류에서 메모리·인증·오디오 문제를 확인하세요.") from exc
if "diar_pipeline" in globals():
    del diar_pipeline
gc.collect()
torch.cuda.empty_cache()
print("추정 화자 수:", len({r["speaker"] for r in diar_data["regular"]}))
display(pd.DataFrame(diar_data["exclusive"]).head(20))

## 셀 16 — 대사에 화자 연결·결과 저장

단어와 가장 많이 겹치는 단일 화자를 배정합니다. 대응 화자가 없거나 길이가 0이면 `UNASSIGNED`로 남깁니다. 화자 경계를 걸친 단어와 겹친 목소리는 `needs_review`에 표시합니다. 같은 화자의 이어지는 단어를 대사로 묶어 CSV·JSON·화자 번호가 포함된 SRT를 저장합니다. CSV·JSON의 `alignment_failed`와 `timestamp_source`에 정렬 실패 및 임시 시간 여부를 남깁니다. 임시 시간에는 배경음도 포함될 수 있습니다.

In [ ]:
words = read_json(run_dir / "aligned_words.json")
diar_data = read_json(diar_path)
def overlap_seconds(start, end, turn):
    return max(0.0, min(end, turn["end"])-max(start, turn["start"]))
def contains_overlapping_speakers(start, end, turns):
    relevant = [t for t in turns if overlap_seconds(start, end, t) > 0]
    for i, left in enumerate(relevant):
        for right in relevant[i+1:]:
            if left["speaker"] != right["speaker"] and min(end, left["end"], right["end"]) > max(start, left["start"], right["start"]):
                return True
    return False
assigned_words = []
for word in words:
    scores = {}
    for turn in diar_data["exclusive"]:
        amount = overlap_seconds(word["start"], word["end"], turn)
        if amount > 0:
            scores[turn["speaker"]] = scores.get(turn["speaker"], 0.0)+amount
    speaker = max(scores, key=scores.get) if scores else "UNASSIGNED"
    overlap = contains_overlapping_speakers(word["start"], word["end"], diar_data["regular"])
    span = word["end"]-word["start"]
    coverage = scores.get(speaker, 0.0)/span if span > 0 else 0.0
    assigned_words.append({**word, "speaker": speaker, "overlap": overlap, "speaker_coverage": coverage,
        "alignment_failed": word.get("alignment_failed", False),
        "timestamp_source": word.get("timestamp_source", "forced_alignment"),
        "needs_review": word["needs_review"] or overlap or len(scores) > 1 or coverage < MIN_SPEAKER_COVERAGE})
save_json(run_dir / "speaker_words.json", assigned_words)
dialogues = []
for word in assigned_words:
    prev = dialogues[-1] if dialogues else None
    join = (prev is not None and prev["speaker"] == word["speaker"] and prev["segment_id"] == word["segment_id"]
        and word["start"]-prev["end"] <= DIALOGUE_GAP_SECONDS and word["end"]-prev["start"] <= DIALOGUE_MAX_SECONDS
        and not prev["text"].rstrip().endswith(("。", "！", "？", "!", "?")))
    if join:
        prev["end"] = max(prev["end"], word["end"])
        prev["text"] += word["display_text"]
        prev["overlap"] |= word["overlap"]
        prev["needs_review"] |= word["needs_review"]
        prev["alignment_failed"] |= word["alignment_failed"]
        if prev["timestamp_source"] != word["timestamp_source"]:
            prev["timestamp_source"] = "mixed"
    else:
        dialogues.append({"start": word["start"], "end": word["end"], "speaker": word["speaker"],
            "text": word["display_text"], "segment_id": word["segment_id"],
            "overlap": word["overlap"], "needs_review": word["needs_review"],
            "alignment_failed": word["alignment_failed"], "timestamp_source": word["timestamp_source"]})
save_json(run_dir / "dialogues.json", dialogues)
dialogue_df = pd.DataFrame(dialogues, columns=["start", "end", "speaker", "text", "segment_id", "overlap", "needs_review",
    "alignment_failed", "timestamp_source"])
dialogue_df.to_csv(run_dir / "dialogues.csv", index=False, encoding="utf-8-sig")
def srt_time(seconds):
    milliseconds = round(seconds*1000)
    hours, milliseconds = divmod(milliseconds, 3600000)
    minutes, milliseconds = divmod(milliseconds, 60000)
    seconds, milliseconds = divmod(milliseconds, 1000)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d},{milliseconds:03d}"
srt_rows, skipped_srt = [], []
for row in dialogues:
    if round(row["end"]*1000) <= round(row["start"]*1000):
        skipped_srt.append(row)
        continue
    srt_rows.append(f"{len(srt_rows)+1}\n{srt_time(row['start'])} --> {srt_time(row['end'])}\n[{row['speaker']}] {row['text']}\n")
(run_dir / "dialogues.srt").write_text("\n".join(srt_rows), encoding="utf-8")
save_json(run_dir / "srt_skipped_zero_duration.json", skipped_srt)
print(f"완료: 대사 {len(dialogues)}개 / 확인 필요 {sum(r['needs_review'] for r in dialogues)}개 / SRT 제외 {len(skipped_srt)}개")
print("결과 폴더:", run_dir)
display(dialogue_df.head(30))

## 셀 17 — 화자별 대표 음성 듣기

각 화자의 긴 단독 발화 구간을 하나씩 재생합니다. 캐릭터 이름을 연결할 때 참고하세요. 같은 캐릭터가 여러 번호로 나뉘거나 여러 캐릭터가 하나로 묶일 수 있습니다.

In [ ]:
SPEAKER_PREVIEW_SECONDS = 10.0
regular = read_json(diar_path)["regular"]
for speaker in sorted({r["speaker"] for r in regular}):
    candidates = [r for r in regular if r["speaker"] == speaker
        and not contains_overlapping_speakers(r["start"], r["end"], regular)]
    if not candidates:
        print(speaker, "단독 구간 없음 — 겹친 발화 JSON을 확인하세요.")
        continue
    turn = max(candidates, key=lambda r: r["end"]-r["start"])
    end = min(turn["end"], turn["start"]+SPEAKER_PREVIEW_SECONDS)
    print(f"{speaker}: {turn['start']:.2f}~{end:.2f}초")
    display(Audio(audio[int(turn["start"]*SR):int(end*SR)], rate=SR))

## 오류를 확인할 위치

| 단계 | 확인할 항목 |
|---|---|
| 셀 1·3 | 설치 로그, 세션 재시작, GPU 배정 |
| 셀 4~6 | Drive 경로, 일본어 트랙, FFmpeg 오류 |
| 셀 7 | 음성 구간 수, VAD가 놓친 작은 목소리 |
| 셀 8~10 | 모델 다운로드, CUDA 메모리, 표시된 전사 구간 |
| 셀 11~13 | 오류 구간의 전사·음성, 0초 정렬 |
| 셀 14 | 모델 이용 조건 동의, 토큰 읽기 권한 |
| 셀 15 | 영상 전체 화자 분리 메모리와 오류 |
| 셀 16·17 | 미배정 화자, 겹친 대사, 잘못 묶인 화자 |

`last_error.json`은 마지막 실패 단계·구간을 기록합니다. 해당 셀의 traceback에 상세 원인이 표시됩니다. CUDA 메모리가 부족하면 세션을 다시 시작하고 같은 `RUN_NAME`으로 재개하세요. 첫 구간에서도 반복되면 오류와 GPU 정보를 먼저 확인하세요.

주요 결과는 `dialogues.csv`, `dialogues.json`, `dialogues.srt`입니다. 원문은 `transcripts.json`, 단어별 시간·화자는 `speaker_words.json`, 겹친 발화의 전체 구간은 `diarization.json`에 있습니다. 0초 대사는 CSV·JSON에 남기고 SRT에서는 제외합니다.

참조: [Qwen 공식 API](https://github.com/QwenLM/Qwen3-ASR), [Community-1 사용법](https://huggingface.co/pyannote/speaker-diarization-community-1), [TorchCodec 호환표](https://github.com/pytorch/torchcodec#installing-torchcodec), [Colab 자원 조건](https://research.google.com/colaboratory/faq.html).

Colab GPU에서의 실제 실행은 아직 확인되지 않았습니다.